# Продолжаем говорить про трансформеры

## Сначала смотрим презу (лежит рядышком)

## Теперь пишем Llama3

### Архитектура T5 (Pre-Norm)

![T5 Architecture](t5_architecture.png)

In [1]:
import json
import math
import os
import glob
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


### Config

In [3]:
@dataclass
class LlamaConfig:
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: int
    rms_norm_eps: float = 1e-6
    max_position_embeddings: int = 2048
    rope_theta: float = 10000.0
    rope_scaling: Optional[dict] = None
    tie_word_embeddings: bool = True

### RMS Norm

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(var + self.eps)
        return x * self.weight

### RoPE

In [5]:
def _build_inv_freq(dim: int, base: float, device=None):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, device=device, dtype=torch.float32) / dim))
    return inv_freq


def _apply_llama3_rope_scaling(inv_freq: torch.Tensor, rope_scaling: dict) -> torch.Tensor:
    if rope_scaling is None:
        return inv_freq
    rope_type = rope_scaling.get("rope_type") or rope_scaling.get("type")
    if rope_type != "llama3":
        return inv_freq

    factor = float(rope_scaling["factor"])
    low_freq_factor = float(rope_scaling.get("low_freq_factor", 1.0))
    high_freq_factor = float(rope_scaling.get("high_freq_factor", 4.0))
    old_ctx = float(rope_scaling.get("original_max_position_embeddings", 8192))

    wavelen = (2.0 * math.pi) / inv_freq
    low_wavelen = old_ctx / low_freq_factor
    high_wavelen = old_ctx / high_freq_factor

    scaled = inv_freq / factor
    orig = inv_freq

    alpha = (wavelen - high_wavelen) / max(low_wavelen - high_wavelen, 1e-6)
    alpha = alpha.clamp(0.0, 1.0)

    mixed = orig * (1.0 - alpha) + scaled * alpha
    out = torch.where(wavelen < high_wavelen, orig, torch.where(wavelen > low_wavelen, scaled, mixed))
    return out


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, base: float, max_position: int, rope_scaling: Optional[dict] = None):
        super().__init__()
        inv_freq = _build_inv_freq(head_dim, base)
        inv_freq = _apply_llama3_rope_scaling(inv_freq, rope_scaling)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        self._cos_cached = None
        self._sin_cached = None
        self._seq_len_cached = 0

    def _build_cache(self, seq_len: int, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)  # [S, Hd/2]
        emb = torch.cat([freqs, freqs], dim=-1)            # [S, Hd]
        self._cos_cached = emb.cos().to(dtype=dtype)
        self._sin_cached = emb.sin().to(dtype=dtype)
        self._seq_len_cached = seq_len

    def forward(self, seq_len: int, device, dtype):
        if self._cos_cached is None or seq_len > self._seq_len_cached or self._cos_cached.device != device:
            self._build_cache(seq_len, device, dtype)
        return self._cos_cached[:seq_len], self._sin_cached[:seq_len]


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor, position_ids: torch.Tensor):
    # cos/sin: [S, Hd], position_ids: [B,T]
    cos_pos = cos[position_ids].unsqueeze(1)  # [B,1,T,Hd]
    sin_pos = sin[position_ids].unsqueeze(1)
    q_out = (q * cos_pos) + (rotate_half(q) * sin_pos)
    k_out = (k * cos_pos) + (rotate_half(k) * sin_pos)
    return q_out, k_out

### KV Cache

In [6]:
class KVCache:
    """
    keys[layer], values[layer]: [B, n_kv, max_seq_len, head_dim]
    """
    def __init__(
        self,
        num_layers: int,
        batch_size: int,
        num_kv_heads: int,
        head_dim: int,
        max_seq_len: int,
        device,
        dtype,
    ):
        self.num_layers = num_layers
        self.batch_size = batch_size
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.device = device
        self.dtype = dtype

        self.keys = [
            torch.empty(batch_size, num_kv_heads, max_seq_len, head_dim, device=device, dtype=dtype)
            for _ in range(num_layers)
        ]
        self.values = [
            torch.empty(batch_size, num_kv_heads, max_seq_len, head_dim, device=device, dtype=dtype)
            for _ in range(num_layers)
        ]
        self.cur_len = 0

    def reset(self):
        self.cur_len = 0

    def append(self, layer_idx: int, k: torch.Tensor, v: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, n_kv, T, Hd = k.shape
        assert B == self.batch_size and n_kv == self.num_kv_heads and Hd == self.head_dim
        end = self.cur_len + T
        if end > self.max_seq_len:
            raise RuntimeError(f"KVCache overflow: need {end}, max_seq_len={self.max_seq_len}")

        self.keys[layer_idx][:, :, self.cur_len:end, :] = k
        self.values[layer_idx][:, :, self.cur_len:end, :] = v

        full_k = self.keys[layer_idx][:, :, :end, :]
        full_v = self.values[layer_idx][:, :, :end, :]
        return full_k, full_v

    def advance(self, n: int):
        self.cur_len += n

    def reorder(self, new_order: torch.Tensor):
        """
        new_order: [B_new] indices into batch dim (B*num_beams)
        """
        for i in range(self.num_layers):
            self.keys[i] = self.keys[i].index_select(0, new_order)
            self.values[i] = self.values[i].index_select(0, new_order)
        self.batch_size = int(new_order.numel())


def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    if n_rep == 1:
        return x
    return x.repeat_interleave(n_rep, dim=1)

### Attention

In [ ]:
class LlamaAttention(nn.Module):
    def __init__(self, cfg: LlamaConfig):
        super().__init__()
        self.hidden_size = cfg.hidden_size
        self.num_heads = cfg.num_attention_heads
        self.num_kv_heads = cfg.num_key_value_heads
        self.head_dim = cfg.hidden_size // cfg.num_attention_heads
        assert self.head_dim * self.num_heads == self.hidden_size

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

        self.rotary_emb = RotaryEmbedding(
            head_dim=self.head_dim,
            base=cfg.rope_theta,
            max_position=cfg.max_position_embeddings,
            rope_scaling=cfg.rope_scaling,
        )

    def forward(
        self,
        x: torch.Tensor,                    # [B,T,H]
        position_ids: torch.Tensor,         # [B,T]
        layer_idx: int,
        kv_cache: Optional[KVCache] = None,
    ):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)       # [B,nH,T,Hd]
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)    # [B,nKv,T,Hd]
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        past_len = 0 if kv_cache is None else kv_cache.cur_len
        total_len = past_len + T
        cos, sin = self.rotary_emb(total_len, device=x.device, dtype=x.dtype)
        q, k = apply_rope(q, k, cos, sin, position_ids)

        if kv_cache is not None:
            full_k, full_v = kv_cache.append(layer_idx, k, v)  # full: [B,nKv,K,Hd]
        else:
            full_k, full_v = k, v

        # GQA expand
        n_rep = self.num_heads // self.num_kv_heads
        k_rep = repeat_kv(full_k, n_rep)  # [B,nH,K,Hd]
        v_rep = repeat_kv(full_v, n_rep)

        scale = 1.0 / math.sqrt(self.head_dim)
        scores = torch.matmul(q, k_rep.transpose(-2, -1)) * scale
        K = scores.shape[-1]

        key_pos = torch.arange(K, device=x.device).view(1, K)                 # [1,K]
        q_pos = (past_len + torch.arange(T, device=x.device)).view(T, 1)      # [T,1]
        causal = key_pos > q_pos                                              # [T,K] True => mask
        scores = scores.masked_fill(causal.view(1, 1, T, K), float("-inf"))

        probs = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v_rep)  # [B,nH,T,Hd]

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim)
        out = self.o_proj(attn_out)
        return out

### FFN

In [8]:
class LlamaMLP(nn.Module):
    def __init__(self, cfg: LlamaConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.up_proj = nn.Linear(cfg.hidden_size, cfg.intermediate_size, bias=False)
        self.down_proj = nn.Linear(cfg.intermediate_size, cfg.hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

### Всё вместе

In [ ]:
class LlamaDecoderLayer(nn.Module):
    def __init__(self, cfg: LlamaConfig):
        super().__init__()
        self.self_attn = LlamaAttention(cfg)
        self.mlp = LlamaMLP(cfg)
        self.input_layernorm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)

    def forward(self, x, position_ids, layer_idx: int, kv_cache: Optional[KVCache] = None):
        h = self.input_layernorm(x)
        attn_out = self.self_attn(h, position_ids=position_ids, layer_idx=layer_idx, kv_cache=kv_cache)
        x = x + attn_out

        h = self.post_attention_layernorm(x)
        x = x + self.mlp(h)
        return x


class LlamaModel(nn.Module):
    def __init__(self, cfg: LlamaConfig):
        super().__init__()
        self.cfg = cfg
        self.embed_tokens = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.layers = nn.ModuleList([LlamaDecoderLayer(cfg) for _ in range(cfg.num_hidden_layers)])
        self.norm = RMSNorm(cfg.hidden_size, cfg.rms_norm_eps)

    def forward(
        self,
        input_ids: torch.Tensor,            # [B,T]
        position_ids: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
    ):
        B, T = input_ids.shape
        if position_ids is None:
            past_len = 0 if kv_cache is None else kv_cache.cur_len
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.embed_tokens(input_ids)
        for i, layer in enumerate(self.layers):
            x = layer(x, position_ids=position_ids, layer_idx=i, kv_cache=kv_cache)
        x = self.norm(x)

        if kv_cache is not None:
            kv_cache.advance(T)
        return x


class LlamaForCausalLM(nn.Module):
    def __init__(self, cfg: LlamaConfig):
        super().__init__()
        self.cfg = cfg
        self.model = LlamaModel(cfg)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor, position_ids: Optional[torch.Tensor] = None, kv_cache: Optional[KVCache] = None):
        h = self.model(input_ids=input_ids, position_ids=position_ids, kv_cache=kv_cache)
        logits = self.lm_head(h)  # [B,T,V]
        return logits

### Подгружаем оригинальную модель

In [12]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/Meta-Llama-3.1-8B", use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Meta-Llama-3.1-8B",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
).cuda()
model.eval()

@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate("Расскажи, что такое трансформеры в NLP простыми словами.\n Трансформеры - это"))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Расскажи, что такое трансформеры в NLP простыми словами.
 Трансформеры - это нейросеть, которая состоит из последовательности слоев. Каждый слой выполняет свою функцию, а затем передает данные следующему слою. Каждый слой может быть как простым, так и сложным, например, это может быть слоя линейной алгебры, слоя полносвязной нейронной сети или слоя с использованием attention.
 
 Давайте рассмотрим простой пример. Предположим, что нам нужно предсказать, является ли человек мужчиной или женщиной на основе его фотографии. Для этого мы можем использовать трансформер. Наша нейросеть будет состоять из нескольких слоев. Первый слой будет принимать фотографию и выдавать ее вектор, который представляет фотографию в пространстве векторов. Этот вектор будет передан в следующий слой, который будет выполнять функцию, подобную линейной алгебре, чтобы получить более высокоуровневый вектор. Этот вектор будет передан в следующий слой, который будет использоваться для предсказания, является ли человек м

In [13]:
my_model = LlamaForCausalLM(model.config)

In [14]:
my_model.load_state_dict(model.state_dict())

<All keys matched successfully>

In [15]:
my_model = my_model.cuda()

### Смотрим на разные сэмплинги

#### Жадный

In [ ]:
@torch.no_grad()
def generate_greedy(
    model: LlamaForCausalLM,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    eos_token_id: Optional[int] = None,
    max_seq_len: Optional[int] = None,
) -> torch.Tensor:
    device = next(model.parameters()).device
    input_ids = input_ids.to(device)

    B, T0 = input_ids.shape
    if max_seq_len is None:
        max_seq_len = T0 + max_new_tokens

    cfg = model.cfg
    kv = KVCache(
        num_layers=cfg.num_hidden_layers,
        batch_size=B,
        num_kv_heads=cfg.num_key_value_heads,
        head_dim=cfg.hidden_size // cfg.num_attention_heads,
        max_seq_len=max_seq_len,
        device=device,
        dtype=next(model.parameters()).dtype,
    )

    logits = model(input_ids, kv_cache=kv)
    next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    out = torch.cat([input_ids, next_id], dim=1)

    finished = torch.zeros(B, dtype=torch.bool, device=device)
    if eos_token_id is not None:
        finished |= (next_id.squeeze(1) == eos_token_id)

    for _ in range(max_new_tokens - 1):
        if eos_token_id is not None and torch.all(finished):
            break
        logits = model(out[:, -1:], kv_cache=kv)  # only last token
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        if eos_token_id is not None:
            next_id = torch.where(finished.unsqueeze(1), torch.full_like(next_id, eos_token_id), next_id)

        out = torch.cat([out, next_id], dim=1)
        if eos_token_id is not None:
            finished |= (next_id.squeeze(1) == eos_token_id)

    return out

#### Top-K

In [17]:
def top_k_top_p_filtering(
    logits: torch.Tensor,
    top_k: int = 0,
    top_p: float = 1.0,
    min_tokens_to_keep: int = 1,
) -> torch.Tensor:
    """
    logits: [B,V]
    returns filtered logits with -inf masked
    """
    B, V = logits.shape
    filtered = logits

    if top_k > 0:
        top_k = min(max(top_k, min_tokens_to_keep), V)
        kth_vals = torch.topk(filtered, top_k, dim=-1).values[:, -1].unsqueeze(-1)
        filtered = torch.where(filtered < kth_vals, torch.full_like(filtered, float("-inf")), filtered)

    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(filtered, descending=True, dim=-1)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumprobs = torch.cumsum(sorted_probs, dim=-1)

        sorted_mask = cumprobs > top_p
        if min_tokens_to_keep > 1:
            sorted_mask[:, :min_tokens_to_keep] = False

        # shift right to keep the first token above top_p as well
        sorted_mask[:, 1:] = sorted_mask[:, :-1].clone()
        sorted_mask[:, 0] = False

        mask = torch.zeros_like(filtered, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_idx, src=sorted_mask)
        filtered = filtered.masked_fill(mask, float("-inf"))

    return filtered


@torch.no_grad()
def generate_topk_topp(
    model: LlamaForCausalLM,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int = 0,
    top_p: float = 1.0,
    eos_token_id: Optional[int] = None,
    max_seq_len: Optional[int] = None,
) -> torch.Tensor:
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    input_ids = input_ids.to(device)

    B, T0 = input_ids.shape
    if max_seq_len is None:
        max_seq_len = T0 + max_new_tokens

    cfg = model.cfg
    kv = KVCache(
        num_layers=cfg.num_hidden_layers,
        batch_size=B,
        num_kv_heads=cfg.num_key_value_heads,
        head_dim=cfg.hidden_size // cfg.num_attention_heads,
        max_seq_len=max_seq_len,
        device=device,
        dtype=dtype,
    )

    logits = model(input_ids, kv_cache=kv)[:, -1, :]  # [B,V]

    out = input_ids
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_new_tokens):
        if eos_token_id is not None and torch.all(finished):
            break

        step_logits = logits
        if temperature != 1.0:
            step_logits = step_logits / max(temperature, 1e-6)

        step_logits = top_k_top_p_filtering(step_logits, top_k=top_k, top_p=top_p)
        probs = torch.softmax(step_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # [B,1]

        if eos_token_id is not None:
            next_id = torch.where(finished.unsqueeze(1), torch.full_like(next_id, eos_token_id), next_id)

        out = torch.cat([out, next_id], dim=1)
        if eos_token_id is not None:
            finished |= (next_id.squeeze(1) == eos_token_id)

        # next logits
        logits = model(next_id, kv_cache=kv)[:, -1, :]

    return out

#### Beam Search

In [18]:
@torch.no_grad()
def generate_beam_search(
    model: LlamaForCausalLM,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    num_beams: int = 4,
    length_penalty: float = 1.0,
    eos_token_id: Optional[int] = None,
    max_seq_len: Optional[int] = None,
) -> torch.Tensor:
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    input_ids = input_ids.to(device)

    B, T0 = input_ids.shape
    if max_seq_len is None:
        max_seq_len = T0 + max_new_tokens

    cfg = model.cfg

    input_ids_exp = input_ids.repeat_interleave(num_beams, dim=0)

    kv = KVCache(
        num_layers=cfg.num_hidden_layers,
        batch_size=B * num_beams,
        num_kv_heads=cfg.num_key_value_heads,
        head_dim=cfg.hidden_size // cfg.num_attention_heads,
        max_seq_len=max_seq_len,
        device=device,
        dtype=dtype,
    )

    logits = model(input_ids_exp, kv_cache=kv)[:, -1, :]
    logprobs = torch.log_softmax(logits, dim=-1)

    beam_scores = torch.zeros(B, num_beams, device=device, dtype=torch.float32)
    beam_scores[:, 1:] = -1e9
    beam_scores = beam_scores.view(B * num_beams)

    seqs = input_ids_exp

    finished = torch.zeros(B * num_beams, dtype=torch.bool, device=device)

    for _ in range(max_new_tokens):
        if eos_token_id is not None:
            logprobs = torch.where(
                finished.unsqueeze(1),
                torch.full_like(logprobs, -1e9),
                logprobs
            )
            if eos_token_id is not None:
                logprobs[finished, eos_token_id] = 0.0

        next_scores = logprobs + beam_scores.unsqueeze(1)
        V = next_scores.shape[-1]

        next_scores = next_scores.view(B, num_beams * V)

        topk_scores, topk_idx = torch.topk(next_scores, k=num_beams, dim=-1)
        next_beam_idx = topk_idx // V
        next_token_id = topk_idx % V

        batch_offsets = (torch.arange(B, device=device) * num_beams).unsqueeze(1)
        reorder = (batch_offsets + next_beam_idx).reshape(B * num_beams)

        kv.reorder(reorder)
        seqs = seqs.index_select(0, reorder)
        finished = finished.index_select(0, reorder)
        beam_scores = topk_scores.reshape(B * num_beams)

        next_token_flat = next_token_id.reshape(B * num_beams, 1)
        seqs = torch.cat([seqs, next_token_flat], dim=1)

        if eos_token_id is not None:
            finished |= (next_token_flat.squeeze(1) == eos_token_id)

        logits = model(next_token_flat, kv_cache=kv)[:, -1, :]
        logprobs = torch.log_softmax(logits, dim=-1)

    # mb normalize?
    final_scores = beam_scores.view(B, num_beams)

    best = torch.argmax(final_scores, dim=-1)
    best_global = (torch.arange(B, device=device) * num_beams + best)
    best_seqs = seqs.index_select(0, best_global)
    return best_seqs


In [19]:
prompt = (
    "Explain the difference between greedy decoding, top-k/top-p sampling, and beam search.\n"
    "For each method, give one situation where it works best.\n"
    "Answer briefly, using bullet points.\n\n"
    "Answer:\n"
)

enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
input_ids = enc["input_ids"].cuda()

eos_id = tokenizer.eos_token_id

max_new = 180

out_greedy = generate_greedy(
    my_model,
    input_ids,
    max_new_tokens=max_new,
    eos_token_id=eos_id,
)

out_topk = generate_topk_topp(
    my_model,
    input_ids,
    max_new_tokens=max_new,
    temperature=0.8,
    top_k=50,
    top_p=1.0,
    eos_token_id=eos_id,
)

out_topp = generate_topk_topp(
    my_model,
    input_ids,
    max_new_tokens=max_new,
    temperature=0.8,
    top_k=0,
    top_p=0.95,
    eos_token_id=eos_id,
)

out_beam = generate_beam_search(
    my_model,
    input_ids,
    max_new_tokens=max_new,
    num_beams=4,
    length_penalty=1.0,
    eos_token_id=eos_id,
)

In [ ]:
prompt_len = input_ids.shape[1]

print("\n=== PROMPT ===\n")
print(prompt)

print("\n=== GREEDY ===\n")
print(tokenizer.decode(out_greedy[0, prompt_len:], skip_special_tokens=True))

print("\n=== TOP-K ===\n")
print(tokenizer.decode(out_topk[0, prompt_len:], skip_special_tokens=True))

print("\n=== TOP-P ===\n")
print(tokenizer.decode(out_topp[0, prompt_len:], skip_special_tokens=True))

print("\n=== BEAM SEARCH ===\n")
print(tokenizer.decode(out_beam[0, prompt_len:], skip_special_tokens=True))


=== PROMPT ===

Explain the difference between greedy decoding, top-k/top-p sampling, and beam search.
For each method, give one situation where it works best.
Answer briefly, using bullet points.

Answer:


=== GREEDY ===

Greedy decoding is a simple and fast decoding method that selects the most likely word at each step. It is suitable for tasks where the output is a sequence of words, such as machine translation or text summarization. However, it can produce suboptimal results when the output is a sequence of tokens, such as in image captioning or speech recognition.

Top-k/top-p sampling is a more sophisticated decoding method that selects the most likely words or tokens based on a probability distribution. It is suitable for tasks where the output is a sequence of tokens, such as image captioning or speech recognition. However, it can be slow and may produce suboptimal results when the output is a sequence of words.

Beam search is a hybrid decoding method that combines the advan

### Сравнение с KV Cache и без

In [21]:
@torch.no_grad()
def generate_greedy_no_cache(
    model: LlamaForCausalLM,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    eos_token_id: Optional[int] = None,
    max_seq_len: Optional[int] = None,
) -> torch.Tensor:
    device = next(model.parameters()).device
    input_ids = input_ids.to(device)

    B, T0 = input_ids.shape
    if max_seq_len is None:
        max_seq_len = T0 + max_new_tokens

    out = input_ids
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_new_tokens):
        if eos_token_id is not None and torch.all(finished):
            break

        logits = model(out)
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

        if eos_token_id is not None:
            next_id = torch.where(finished.unsqueeze(1), torch.full_like(next_id, eos_token_id), next_id)
        
        out = torch.cat([out, next_id], dim=1)
        if eos_token_id is not None:
            finished |= (next_id.squeeze(1) == eos_token_id)

    return out

In [23]:
out_greedy_no_cache = generate_greedy_no_cache(
    my_model,
    input_ids,
    max_new_tokens=max_new,
    eos_token_id=eos_id,
)

In [25]:
print(tokenizer.decode(out_greedy_no_cache[0, input_ids.shape[1]:], skip_special_tokens=True))

Greedy decoding is a simple and fast decoding method that selects the most likely word at each step. It is suitable for tasks where the output is a sequence of words, such as machine translation or text summarization. However, it can produce suboptimal results when the output is a sequence of tokens, such as in image captioning or speech recognition.

Top-k/top-p sampling is a more sophisticated decoding method that selects the most likely words or tokens based on a probability distribution. It is suitable for tasks where the output is a sequence of tokens, such as image captioning or speech recognition. However, it can be slow and may produce suboptimal results when the output is a sequence of words.

Beam search is a hybrid decoding method that combines the advantages of greedy decoding and top-k/top-p sampling. It is suitable for tasks where the output is a sequence of words or tokens, such as


In [28]:
def benchmark_kvcache(
        generate_func,
        model: LlamaForCausalLM,
        input_ids: torch.Tensor,
        max_new_tokens: int,
        eos_token_id: Optional[int] = None,
        max_seq_len: Optional[int] = None,
        warmup=1,
        iter=3
    ):

    for _ in range(warmup):
        _ = generate_func(
            model,
            input_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=eos_token_id,
            max_seq_len=max_seq_len,
        )
    
    start = time.time()
    
    for _ in range(iter):
        _ = generate_func(
            model,
            input_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=eos_token_id,
            max_seq_len=max_seq_len,
        )
        torch.cuda.synchronize()
    
    end = time.time()

    return (end - start) / iter

In [29]:
time_with_kvcache = benchmark_kvcache(
    generate_greedy,
    my_model,
    input_ids,
    max_new_tokens=max_new,
    eos_token_id=eos_id,
)

time_without_kvcache = benchmark_kvcache(
    generate_greedy_no_cache,
    my_model,
    input_ids,
    max_new_tokens=max_new,
    eos_token_id=eos_id,
)


print(f"time_with_kvcache={time_with_kvcache*1000:.1f} ms")
print(f"time_without_kvcache={time_without_kvcache*1000:.1f} ms")

time_with_kvcache=6814.8 ms
time_without_kvcache=27722.5 ms


Но при этом мы жервуем памятью в размере 2 * `num_layers` * `n_kv_heads` * `head_dim` * B * `max_seq_len` * `bytes_per_element`, где `bytes_per_element` в нашем случае 4, но вообще 2 (если гонять в bf16)

In [43]:
print((2 * my_model.cfg.num_hidden_layers * my_model.cfg.num_key_value_heads * my_model.cfg.head_dim * (max_new + input_ids.shape[1]) * 4) / 1e9, "GB")

0.05767168 GB
